In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from tqdm import tqdm
import os
os.chdir('E:/Machine_Learning/Cases/Medical_Cost_Personal/')

In [2]:
df = pd.read_csv('insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
df.dtypes

age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object

In [4]:
X,y = df.drop('charges',axis = 1),df['charges']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.3,random_state=26)

ohe = OneHotEncoder(sparse_output= False,drop='first')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')
X_train_ohe = ct.fit_transform(X_train)
X_test_ohe = ct.transform(X_test)

In [5]:
print(X_train_ohe.shape, y_train.shape)
print(X_train_ohe.dtypes)
print(y_train.dtype)

(936, 8) (936,)
sex_male            float64
smoker_yes          float64
region_northwest    float64
region_southeast    float64
region_southwest    float64
age                   int64
bmi                 float64
children              int64
dtype: object
float64


## Gradient Boosting

In [6]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            gbm = GradientBoostingRegressor(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            gbm.fit(X_train_ohe,y_train)
            y_pred = gbm.predict(X_test_ohe)
            scores.append([l,m,n,r2_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','r2'])
df_scores.sort_values(by='r2',ascending=False)

100%|██████████| 10/10 [00:12<00:00,  1.21s/it]


,Learning_rate,Max_depth,n_estimators,r2
19,0.108889,3.0,50,0.869232
30,0.207778,3.0,25,0.869119
20,0.108889,3.0,100,0.868956
18,0.108889,3.0,25,0.866720
42,0.306667,3.0,25,0.866690
...,...,...,...,...
4,0.010000,2.0,50,0.522892
9,0.010000,4.0,25,0.344310
0,0.010000,NaN,25,0.339508
6,0.010000,3.0,25,0.338345


## XGBoosting

In [7]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            xgb = XGBRegressor(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            xgb.fit(X_train_ohe,y_train)
            y_pred = xgb.predict(X_test_ohe)
            scores.append([l,m,n,r2_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','r2'])
df_scores.sort_values(by='r2',ascending=False)

100%|██████████| 10/10 [00:15<00:00,  1.52s/it]


,Learning_rate,Max_depth,n_estimators,r2
19,0.108889,3.0,50,0.872067
30,0.207778,3.0,25,0.871976
39,0.306667,2.0,25,0.871520
22,0.108889,4.0,50,0.870778
33,0.207778,4.0,25,0.870762
...,...,...,...,...
4,0.010000,2.0,50,0.522753
0,0.010000,NaN,25,0.339735
9,0.010000,4.0,25,0.338969
6,0.010000,3.0,25,0.335574


## Light GBoost

In [8]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            lgb = LGBMRegressor(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            lgb.fit(X_train_ohe,y_train)
            y_pred = lgb.predict(X_test_ohe)
            scores.append([l,m,n,r2_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','r2'])
df_scores.sort_values(by='r2',ascending=False)

  0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000195 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 936, number of used features: 8
[LightGBM] [Info] Start training from score 13106.545532
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000161 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 936, number of used features: 8
[LightGBM] [Info] Start training from score 13106.545532
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000157 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 936, number of used features: 8
[LightGBM] [Info] Start trainin

 10%|█         | 1/10 [00:00<00:02,  3.71it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000226 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 936, number of used features: 8
[LightGBM] [Info] Start training from score 13106.545532
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

 20%|██        | 2/10 [00:00<00:02,  3.85it/s]

...

 90%|█████████ | 9/10 [00:02<00:00,  3.88it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000157 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 936, number of used features: 8
[LightGBM] [Info] Start training from score 13106.545532
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

100%|██████████| 10/10 [00:02<00:00,  3.71it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000217 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 936, number of used features: 8
[LightGBM] [Info] Start training from score 13106.545532
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

,Learning_rate,Max_depth,n_estimators,r2
30,0.207778,3.0,25,0.870819
31,0.207778,3.0,50,0.869950
19,0.108889,3.0,50,0.869713
17,0.108889,2.0,100,0.869398
22,0.108889,4.0,50,0.869261
...,...,...,...,...
4,0.010000,2.0,50,0.522892
0,0.010000,NaN,25,0.347231
9,0.010000,4.0,25,0.344334
6,0.010000,3.0,25,0.338345


## Cat Boost

In [9]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            cat = CatBoostRegressor(random_state=26,learning_rate=l,verbose=False,max_depth=m,n_estimators=n)
            cat.fit(X_train_ohe,y_train)
            y_pred = cat.predict(X_test_ohe)
            scores.append([l,m,n,r2_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','r2'])
df_scores.sort_values(by='r2',ascending=False)

100%|██████████| 10/10 [00:21<00:00,  2.15s/it]


,Learning_rate,Max_depth,n_estimators,r2
31,0.207778,3.0,50,0.874315
99,0.801111,2.0,25,0.873775
78,0.603333,3.0,25,0.873340
34,0.207778,4.0,50,0.873296
32,0.207778,3.0,100,0.872602
...,...,...,...,...
4,0.010000,2.0,50,0.460703
9,0.010000,4.0,25,0.303635
6,0.010000,3.0,25,0.302483
0,0.010000,NaN,25,0.290878


# Catboost Without Encoding Using cat_features

In [10]:
catagorical = ['sex','smoker','region']
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            cat = CatBoostRegressor(random_state=26,learning_rate=l,verbose=False,max_depth=m,n_estimators=n,cat_features=catagorical)
            cat.fit(X_train,y_train)
            y_pred = cat.predict(X_test)
            scores.append([l,m,n,r2_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','r2'])
df_scores.sort_values(by='r2',ascending=False)

100%|██████████| 10/10 [01:46<00:00, 10.63s/it]


,Learning_rate,Max_depth,n_estimators,r2
20,0.108889,3.0,100,0.873361
23,0.108889,4.0,100,0.871817
42,0.306667,3.0,25,0.871504
36,0.306667,NaN,25,0.871146
14,0.108889,NaN,100,0.871091
...,...,...,...,...
4,0.010000,2.0,50,0.465715
6,0.010000,3.0,25,0.300450
9,0.010000,4.0,25,0.296912
0,0.010000,NaN,25,0.293823
